# Notebook 2 — Data Acquisition

What we set up here:

1. Define our two AOI polygons (Ebro Delta + Maresme), save them as GeoJSON.
2. Search Sentinel-2 L2A scenes via STAC for the **2017–2025 trend window**.
3. Search a narrower window around **Storm Gloria (Jan 2020)** for pre/post imagery.
4. Build a lazy `xarray` stack with `stackstac` (no GBs are pulled here — that happens later, on demand).
5. Document the access patterns for **ICGC LiDAR**, **CMEMS waves**, and **CHE Ebro discharge**.

**The principle:** *search and inventory now, materialise only when needed.* Sentinel-2 is small per scene but big in aggregate; we should know exactly what we'd be loading before we load it.


## Data layers we need

| Layer | Source | Why | Access pattern |
|---|---|---|---|
| Sentinel-2 L2A | Microsoft Planetary Computer (STAC) | GFM input + classical (CoastSat) input | Lazy via `pystac-client` + `stackstac` |
| Sentinel-1 SAR | MS Planetary Computer | Cloud-blind imagery for Storm Gloria, possible TerraMind input | Same STAC pattern |
| **ICGC MDT 2 m** (LiDAR-derived bare-earth DEM, 3 campaigns) | ICGC public download portal / WCS | Validation truth: shoreline from DEM ∩ tidal datum | Tile download *or* `owslib` WCS subset |
| ICGC orthophotos | ICGC WMS/WMTS | Visual sanity check, possible fine-grained validation | WMS endpoint or portal download |
| Wave climate | CMEMS Mediterranean wave reanalysis | Correlate erosion with storm energy | `copernicusmarine` package (login required) |
| Ebro discharge | CHE / CEDEX Anuario de Aforos | Test sediment-starvation hypothesis at delta | CEDEX website scrape (no JSON API) |
| Tide gauges | Puertos del Estado (Barcelona, L'Ametlla) | Tidal datum for shoreline extraction | Download form |

Storage convention: every download lands under `data/raw/<source>/<aoi_name>/`. Cleaned/clipped derivatives go to `data/interim/`. Analysis-ready products (shorelines, embeddings, masks) go to `data/processed/`.

**A note on "why MDT, not raw LiDAR":** see the dedicated section below — short answer is that the derived DEM is the right product for shoreline-from-elevation, an order of magnitude smaller, and avoids ICGC's per-tile download friction with the raw point clouds.


## A note on STAC catalogs

> **Analogy:** A STAC catalog is a *library card catalog for satellite imagery*. Instead of downloading every book to find what you want, you query the catalog ("all blue-spine books published 2017–2025 about coastlines"), get back metadata cards, then borrow only the books you actually need.

Three flavours we could use:

- **Microsoft Planetary Computer** — anonymous tokens via the `planetary-computer` package, fast, well-maintained. *We use this for prototyping.*
- **Element84 Earth Search** — also open, no auth needed.
- **CDSE (Copernicus Data Space Ecosystem)** — the official ESA home for Sentinel data; requires an account but is the long-term, authoritative source. Switching from PC to CDSE is mostly a URL + auth change.

**Choice for now:** PC. We can switch to CDSE later if a tile is missing or we need more recent acquisitions than PC mirrors.


In [1]:
# Imports + project paths
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import json
import geopandas as gpd
import pandas as pd
import folium

# STAC + lazy stack
import pystac_client
import planetary_computer as pc
import stackstac

from aois import AOIS, DESCRIPTIONS, WAVE_BUOYS  # local helper module

print(f'Project root: {PROJECT_ROOT}')
print(f'AOIs loaded:  {list(AOIS.keys())}')


Project root: /Users/jamesmurphy/Documents/Documents/GitHub/GFM_coasts
AOIs loaded:  ['ebro_delta', 'maresme']


## AOI definitions

Bounding boxes in WGS84 — coordinates and reasoning live in `src/aois.py` as the single source of truth.

| AOI | Bounding box (lon, lat) | Why these bounds |
|---|---|---|
| Ebro Delta | (0.50, 40.55) → (0.95, 40.85) | Captures both hemideltas (Fangar N, Banya S) plus a coastal-water buffer offshore |
| Maresme | (2.25, 41.40) → (2.90, 41.75) | Mongat to Blanes — full extent of the chronic-deficit stretch |

Bboxes are deliberately generous on the inland side. The actual coastal strip we care about is the ~200 m landward and ~500 m seaward of the waterline — we'll clip to that buffer in later notebooks once we have a coastline reference.

**A choice worth flagging:** Bboxes are not the same as polygons of the actual coast. For STAC search this doesn't matter (you only need an `intersects` geometry). For later masking and area calculations, we'll want the ICGC official coastline shapefile.


In [2]:
# Build a GeoDataFrame from the AOI dict and save as GeoJSON for downstream use
gdf = gpd.GeoDataFrame(
    {
        'name': list(AOIS.keys()),
        'description': [DESCRIPTIONS[k] for k in AOIS],
    },
    geometry=list(AOIS.values()),
    crs='EPSG:4326',
)

aoi_dir = PROJECT_ROOT / 'data' / 'aois'
aoi_dir.mkdir(parents=True, exist_ok=True)
out = aoi_dir / 'aois.geojson'
gdf.to_file(out, driver='GeoJSON')
print(f'Wrote {out.relative_to(PROJECT_ROOT)} ({len(gdf)} polygons)')
gdf


Wrote data/aois/aois.geojson (2 polygons)


,name,description,geometry
0,ebro_delta,Ebro Delta — sediment-starved delta with activ...,"POLYGON ((0.95 40.55, 0.95 40.85, 0.5 40.85, 0..."
1,maresme,"Maresme coast — Mongat to Blanes, north of Bar...","POLYGON ((2.9 41.4, 2.9 41.75, 2.25 41.75, 2.2..."


In [4]:
# Quick map — sanity check that the bboxes cover what we expect.
m = folium.Map(location=[41.2, 1.5], zoom_start=8, tiles='OpenStreetMap')

colours = {'ebro_delta': '#d63384', 'maresme': '#0d6efd'}
for _, row in gdf.iterrows():
    folium.GeoJson(
        row.geometry.__geo_interface__,
        name=row['name'],
        tooltip=f"<b>{row['name']}</b><br/>{row['description']}",
        style_function=lambda x, c=colours[row['name']]: {
            'color': c, 'weight': 2, 'fillColor': c, 'fillOpacity': 0.1,
        },
    ).add_to(m)

# Mark the wave buoy locations too
for name, p in WAVE_BUOYS.items():
    folium.CircleMarker(
        location=[p['lat'], p['lon']], radius=5, color='black', fill=True,
        tooltip=f'Wave buoy: {name}',
    ).add_to(m)

folium.LayerControl().add_to(m)
(PROJECT_ROOT / 'maps').mkdir(exist_ok=True)
m.save(str(PROJECT_ROOT / 'maps' / 'aois.html'))
m


## Sentinel-2 search strategy

Decisions worth making explicit:

- **Time range:** `2017-01-01` → `2025-12-31`. S2A flew from June 2015; S2B from July 2017 brought the 5-day revisit. Using 2017 as the start gives us the dual-satellite cadence everywhere.
- **Cloud filter:** `eo:cloud_cover < 30`. Loose for now — we can tighten in analysis. Mediterranean is generally clearer than Atlantic Spain, but winter cloudiness matters for storm windows.
- **Bands we'll eventually pull:** `B02/B03/B04` (RGB), `B08` (NIR), `B11/B12` (SWIR — needed for water indices like MNDWI), and `SCL` (scene classification, for masking).
- **No download yet.** We just inventory the scenes; the lazy `stackstac` step shows the shape of what we *could* materialise.


In [5]:
# Connect to Microsoft Planetary Computer; pc.sign_inplace adds anonymous SAS tokens
# automatically when items are returned, so downstream tools can read the COGs.
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=pc.sign_inplace,
)

def search_trend(aoi_geom):
    return catalog.search(
        collections=['sentinel-2-l2a'],
        intersects=aoi_geom,
        datetime='2017-01-01/2025-12-31',
        query={'eo:cloud_cover': {'lt': 30}},
    ).item_collection()

trend_items = {k: search_trend(geom) for k, geom in AOIS.items()}
for k, items in trend_items.items():
    print(f'{k:<12} {len(items):>5} S2 scenes')


ebro_delta    3814 S2 scenes
maresme        692 S2 scenes


In [6]:
# Inspect what we got: dates, cloud cover, scene density per year.
def items_to_df(items, aoi_name):
    return pd.DataFrame([
        {
            'aoi': aoi_name,
            'datetime': i.datetime,
            'cloud_pct': i.properties.get('eo:cloud_cover'),
            's2_id': i.id,
        }
        for i in items
    ])

df = pd.concat([
    items_to_df(items, k) for k, items in trend_items.items()
], ignore_index=True)
df['year'] = df.datetime.dt.year

print('Scenes per year per AOI (cloud < 30%):')
print(df.groupby(['aoi', 'year']).size().unstack(fill_value=0))
print()
print('Cloud-cover summary:')
print(df.groupby('aoi').cloud_pct.describe()[['mean','50%','max']].round(1))


Scenes per year per AOI (cloud < 30%):
year        2017  2018  2019  2020  2021  2022  2023  2024  2025
aoi                                                             
ebro_delta   317   355   435   370   316   530   657   354   480
maresme       60    56    81    68    50   114   114    65    84

Cloud-cover summary:
            mean  50%   max
aoi                        
ebro_delta   8.2  4.6  30.0
maresme      8.9  6.4  29.8


In [7]:
# Lazy stack demo for the Ebro Delta AOI.
# IMPORTANT: this does not pull pixels — `stack` is a Dask-backed xarray DataArray.
# Materialise per-scene with `.compute()` only when you need pixel values.

ebro_items = trend_items['ebro_delta']

ebro_stack = stackstac.stack(
    ebro_items,
    epsg=32631,                 # UTM 31N — appropriate for Catalan coast
    resolution=10,              # metres
    bounds_latlon=AOIS['ebro_delta'].bounds,
    assets=['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'SCL'],
    chunksize=2048,
)

print(ebro_stack)
print()
print(f'Estimated full materialisation size: '
      f'{ebro_stack.nbytes / 1e9:.1f} GB '
      f'(don\'t do this — subset first)')


<xarray.DataArray 'stackstac-57fc215b26342d4339a7a170d1090514' (time: 3814,
                                                                band: 7,
                                                                y: 3430, x: 3889)> Size: 3TB
dask.array<fetch_raster_window, shape=(3814, 7, 3430, 3889), dtype=float64, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>
Coordinates: (12/43)
  * time                                     (time) datetime64[ns] 31kB 2017-...
    id                                       (time) <U54 824kB 'S2A_MSIL2A_20...
    eo:cloud_cover                           (time) float64 31kB 8.113 ... 3.594
    s2:datatake_id                           (time) <U34 519kB 'GS2A_20170103...
    sat:orbit_state                          (time) <U10 153kB 'descending' ....
    s2:unclassified_percentage               (time) float64 31kB 7.263 ... 0....
    ...                                       ...
    s2:datatake_type                         <U8 32B 'INS-NOBS'
    s2

## Storm Gloria window

Storm Gloria hit Catalonia **19–23 January 2020** with significant wave heights >8 m at the Tarragona buoy and storm surge of ~1 m. We want pre- and post-storm scenes as close to the event as cloud cover allows.

- **Pre window:** 2019-12-15 → 2020-01-18
- **Post window:** 2020-01-23 → 2020-02-29

Cloud filter is loosened to `<60%` because winter clarity is worse and we can pick the cleanest scene from a sparse set.


In [8]:
def search_gloria(aoi_geom, start, end):
    return catalog.search(
        collections=['sentinel-2-l2a'],
        intersects=aoi_geom,
        datetime=f'{start}/{end}',
        query={'eo:cloud_cover': {'lt': 60}},
    ).item_collection()

gloria_windows = {
    'pre':  ('2019-12-15', '2020-01-18'),
    'post': ('2020-01-23', '2020-02-29'),
}

gloria = {}
for aoi_name, geom in AOIS.items():
    for label, (start, end) in gloria_windows.items():
        items = search_gloria(geom, start, end)
        gloria[(aoi_name, label)] = items
        cleanest = min(items, key=lambda i: i.properties.get('eo:cloud_cover', 100), default=None)
        cleanest_str = (
            f"cleanest: {cleanest.datetime.date()} ({cleanest.properties['eo:cloud_cover']:.0f}% cloud)"
            if cleanest is not None else 'no scenes'
        )
        print(f'{aoi_name:<12} {label:>5} {start}→{end}  {len(items):>3} items   {cleanest_str}')


ebro_delta     pre 2019-12-15→2020-01-18   53 items   cleanest: 2020-01-16 (0% cloud)
ebro_delta    post 2020-01-23→2020-02-29   53 items   cleanest: 2020-02-10 (0% cloud)
maresme        pre 2019-12-15→2020-01-18    9 items   cleanest: 2020-01-08 (5% cloud)
maresme       post 2020-01-23→2020-02-29    8 items   cleanest: 2020-02-22 (1% cloud)


## ICGC LiDAR-derived DEM — the right product for our task

ICGC's portal only lets you select tiles one at a time, GeoJSON-upload-for-filtering errors out, and **raw LiDAR is huge** (~1 pt/m² point clouds across both AOIs across 3 campaigns gets you well into the tens of GB before any analysis).

We don't need the point cloud. We need *elevation at the coast* to derive a shoreline (DEM ∩ tidal datum). ICGC publishes that as a derived raster:

- **MDT 2 m** — bare-earth DEM at 2 m, per LiDAR campaign. Per-tile size ~30–80 MB. *Plenty for shoreline-from-DEM at our scale.*
- **MDT 5 m** — same product at 5 m. Per-tile ~5–15 MB. Trend analysis still works; we lose precision on narrow scarps. Used as fallback when a campaign doesn't publish 2 m (the 2008 campaign in particular).

### Why this is the right call

| | Raw LiDAR (LAZ) | MDT 2 m derived raster |
|---|---|---|
| What you control | Ground filtering, intensity, returns | Pre-classified bare earth (ICGC's filter) |
| Per-tile size | ~50–100 MB (compressed LAZ) | ~30–80 MB (GeoTIFF) |
| Tile footprint | 2 × 2 km | **5 × 5 km — fewer tiles** |
| Manual download burden | Many small tiles, one at a time | ~5–10× fewer tiles |
| Read in Python | `pdal` / `laspy` (heavy deps) | `rasterio` / `rioxarray` (already in env) |
| What we lose for shoreline derivation | Custom ground filtering — we wouldn't do this anyway | Nothing |

**Total storage estimate** (3 campaigns × both AOIs at MDT 2 m): roughly **3–6 GB**. Fits on a laptop comfortably.

### Two ways to fetch the MDT

**Option A — tile-by-tile from the portal** (same friction as raw LiDAR, but far fewer tiles):

1. Go to <https://www.icgc.cat/Descarregues/Elevacions> → "Model digital del terreny - 2 m".
2. Select tiles intersecting our AOIs.
3. Save into `data/raw/icgc_mdt/{aoi_name}/{campaign}/`.

**Option B — ICGC WCS endpoint** (preferred — no tile management, gives you exactly the AOI extent):

```python
# Sketch — full implementation in notebook 03.
from owslib.wcs import WebCoverageService

WCS_URL = 'https://geoserveis.icgc.cat/...'   # exact endpoint TBD; see ICGC Geoserveis catalogue
wcs = WebCoverageService(WCS_URL, version='2.0.1')
# wcs.contents lists available coverages, e.g. 'mdt_2m_2016', 'mdt_2m_2020' …
```

`owslib` isn't in the base env yet — we add it to `pyproject.toml` when we actually wire this up in notebook 03 (keeps the base env light).

### A caveat on the campaigns

ICGC doesn't always publish MDT for every LiDAR campaign at the same resolution. The 2008 baseline may only exist at 5 m; the 2016 and recent campaigns are typically at 2 m. We pick what's available per campaign and **report the resolution alongside any RMSE figure** so the comparison stays honest.

**Orthophotos** as a separate visual check: same portal, "Ortofoto Catalunya". WMS endpoint: `https://geoserveis.icgc.cat/icc_ortoinstamaps/wms/service`.


## Wave climate via CMEMS

Wave climate is available from the [Copernicus Marine Service (CMEMS)](https://marine.copernicus.eu) via the `copernicusmarine` Python package.

- **Dataset:** `cmems_mod_med_wav_my_4.2km_PT1H-i` (Mediterranean wave reanalysis, 4.2 km, hourly).
- **Variables:** `VHM0` (significant wave height), `VTM02` (mean period), `VMDR` (direction).
- **Sample points:** Tarragona (offshore Ebro) and Begur (proxy for the Maresme/Costa Brava). Defined in `src/aois.py` as `WAVE_BUOYS`.

**Authentication:** run `copernicusmarine login` once on your machine, or set `COPERNICUSMARINE_SERVICE_USERNAME` / `COPERNICUSMARINE_SERVICE_PASSWORD` env vars (free registration). We deliberately don't bake credentials into the notebook.

**Alternative for Gloria event detail:** Puertos del Estado provides direct buoy records at minute resolution. No clean API; download form at <https://www.puertos.es/es-es/oceanografia>. We'll use CMEMS for the trend (uniform coverage) and Puertos del Estado for the storm zoom.


In [ ]:
# Sketch — only run after `copernicusmarine login` at the terminal.
# We don't execute this in notebook 02; it's here so the call shape is visible.

import copernicusmarine  # noqa: F401

CMEMS_WAVES = {
    'dataset_id': 'cmems_mod_med_wav_my_4.2km_PT1H-i',
    'variables':  ['VHM0', 'VTM02', 'VMDR'],
}

# Example call (commented — credentials required):
# ds = copernicusmarine.subset(
#     dataset_id=CMEMS_WAVES['dataset_id'],
#     variables=CMEMS_WAVES['variables'],
#     minimum_longitude=1.40, maximum_longitude=1.55,
#     minimum_latitude=41.00, maximum_latitude=41.15,
#     start_datetime='2017-01-01', end_datetime='2025-12-31',
#     output_directory='../data/raw/cmems_waves',
# )

print('CMEMS query plan defined; execute in notebook 03 once logged in.')


## Ebro discharge via CHE / CEDEX

Ebro discharge is recorded at gauging stations operated by the **Confederación Hidrográfica del Ebro (CHE)** through its **SAIH** (Sistema Automático de Información Hidrológica) network. The key gauge for delta sediment supply is **Tortosa** (`A027`, ~30 km upstream of the river mouth — the closest large gauge to the delta).

Two sources to know about:

- **SAIH near-real-time:** <http://www.saihebro.com/saihebro/index.php> — last ~24 months, JSON/CSV exports per gauge. Good for the Gloria event detail.
- **CEDEX Anuario de Aforos (historic):** <https://ceh.cedex.es/anuarioaforos/> — full historic records including monthly means. Better for our 2017–2025 trend.

Plan: scrape the CEDEX anuario for Tortosa monthly discharge, save to `data/raw/che_discharge/tortosa_monthly.csv`. We'll do this in **notebook 03**.

**Hypothesis to hold on to:** low-discharge years correlate with faster southern-hemidelta retreat, because reduced fluvial sediment input lets the wave climate erode without replacement. We can test this directly once we have both timeseries.


## What we have

- AOIs defined and saved to `data/aois/aois.geojson`.
- AOI map exported to `maps/aois.html`.
- Sentinel-2 trend inventory (`trend_items` dict) — thousands of scenes, lazily indexed.
- Storm Gloria pre/post inventory (`gloria` dict).
- Lazy `stackstac` stack pattern shown — ready to materialise per-tile when needed.
- Documented access patterns for ICGC LiDAR, CMEMS waves, Puertos del Estado, and CHE discharge.

## What's next

**Notebook 03 — Classical baseline.** We run CoastSat over the trend window for both AOIs, derive shorelines, set up DSAS-style transects, and compute change rates. This gives us the *familiar set of numbers* we'll later place GFM outputs alongside.

Two checks to do before notebook 03:

1. **Run the AOI map cell above** and confirm the bounding boxes capture the coast you care about. The Ebro bbox is generous; the Maresme one is tight.
2. **`copernicusmarine login`** at the terminal so notebook 03 can fetch waves without prompting.

---

**Inspirational questions to take into notebook 03:**

- *Which year, in our 9-year window, would you bet was the worst for the southern Ebro hemidelta?* If we have a guess up front, we can check it against the data when it lands.
- *Is the Maresme story dominated by storms or by gradual sediment deficit?* The answer changes how we read CoastSat results — a method tuned for slow change might miss episodic Maresme losses.
